In [19]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [20]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [21]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 18:58:33, wtch_dt_end:2026-07-21 18:58:33


In [22]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [23]:
@file:DependsOn("org.json:json:20250107")

In [24]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [25]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2130,1078,0,0.260000,61,5.157455,2.455338,0.250000,3.931417,5.500500,6.415417,19.907000
rtmWqChpla,Comparable<*>,2130,1308,0,,179,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2130,1,0,,2130,null,null,,,,,
rtmWqWtchStaCd,String,2130,14,0,SEA1005,179,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2130,2130,0,1,1,1065.500000,615.022357,1,532.916667,1065.500000,1598.083333,2130
rtmWqTu,Int,2130,160,0,5,195,25.609390,33.864642,0,5.000000,11.000000,33.000000,232
ph,Double,2130,141,0,7.500000,53,7.675075,0.295461,7.020000,7.470000,7.630000,7.920000,9.080000
rtmWqSlnty,Number,2130,1971,0,32.705002,4,21.822031,10.286188,0.020000,13.914000,26.645000,29.555000,34.032001
rtmWqCndctv,Float,2130,2047,0,44.272999,3,34.072472,15.387617,0.046000,23.209000,39.844000,45.022499,54.451000
rtmWqWtchDtlDt,String,2130,189,0,2026-07-20 19:10:00.0,14,null,null,2026-07-20 19:00:00.0,2026-07-21 00:55:00.0,2026-07-21 06:55:00.0,2026-07-21 12:40:00.0,2026-07-21 18:30:00.0


In [26]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}.convert {  num and rtmWqTu and rtmWqSlnty }.with { it.toString().trim().toDouble() }


df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2130,1078,0,0.260000,61,5.157455,2.455338,0.250000,3.931417,5.500500,6.415417,19.907000
rtmWqChpla,Double,2130,1308,0,0.000000,179,5.262524,5.287863,0.000000,1.300000,3.059000,7.920000,25.039000
rtmWqWtchStaCd,String,2130,14,0,SEA1005,179,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Double,2130,2130,0,1.000000,1,1065.500000,615.022357,1.000000,532.916667,1065.500000,1598.083333,2130.000000
rtmWqTu,Double,2130,160,0,5.000000,195,25.609390,33.864642,0.000000,5.000000,11.000000,33.000000,232.000000
ph,Double,2130,141,0,7.500000,53,7.675075,0.295461,7.020000,7.470000,7.630000,7.920000,9.080000
rtmWqSlnty,Double,2130,1971,0,32.705002,4,21.822031,10.286188,0.020000,13.919500,26.647000,29.555167,34.032001
rtmWqCndctv,Float,2130,2047,0,44.272999,3,34.072472,15.387617,0.046000,23.209000,39.844000,45.022499,54.451000
rtmWqWtchDtlDt,LocalDateTime,2130,189,0,2026-07-20T19:10,14,null,null,2026-07-20T19:00,2026-07-21T00:55,2026-07-21T06:55,2026-07-21T12:40,2026-07-21T18:30
rtmWtchWtem,Double,2130,751,0,26.990000,13,26.380000,2.333511,19.740000,24.980000,26.459999,28.190001,30.660000


In [27]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [28]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1.000000,2026-07-20T19:00,4.550000,6.980000,SEA5003,14.000000,7.530000,32.646000,49.502998,24.620001
2.000000,2026-07-20T19:00,6.740000,8.390000,SEA7002,9.000000,7.570000,26.792999,39.060001,21.670000
3.000000,2026-07-20T19:00,1.480000,0.000000,SEA1005,14.000000,7.300000,0.532000,1.082000,27.889999
4.000000,2026-07-20T19:00,5.752000,3.124000,NEP2001,8.000000,7.800000,15.290000,25.127001,26.250000
5.000000,2026-07-20T19:00,7.800000,18.760000,SEA6001,46.000000,8.090000,28.915001,44.827999,27.420000


In [29]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="lnPaFn" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("lnPaFn");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845